# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1: Click capture by position

The paper reports that pages in higher search positions had higher weighted CTR,
with the strongest click capture in the top positions.

**My methodology question:** Because this finding is based on observed CTR and
position, I would ask how the comparison controls for other factors that may
differ between position groups, such as query intent, content type, or page
characteristics. I would also clarify that this is an observed relationship
rather than evidence that improving position alone causes the reported CTR
change.

## Finding 2: Growth prediction

The paper reports that its growth model performed at about 90% accuracy on new
pages from brands represented in training, and about 75% on brands not seen
during training.

**My methodology question:** I would ask exactly how the growth label was
created, including the future time window and threshold used to define
growth. I would also ask whether the unseen-brand validation was kept
completely separate from training and model selection. If the goal is to
support generalization to new brands, the unseen-brand split is important
evidence, but the claim should remain limited to the tested dataset and
validation design.

Overall, these questions are meant to make the methodology clearer rather than
to reject the findings.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Load the starter dataset
df = pd.read_csv(
    "https://raw.githubusercontent.com/ArshadNazir1/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
)

# Create the same target setup used in Week 5
df["has_position"] = df["avg_position"] > 0

median_ctr_per_tier = (
    df[df["has_position"]]
    .groupby("position_tier")["ctr"]
    .median()
)

df["expected_ctr"] = df["position_tier"].map(median_ctr_per_tier)

df["ctr_underperforming"] = (
    df["has_position"] &
    (df["ctr"] < df["expected_ctr"])
)

# Only pages with ranking data
model_df = df[df["has_position"]].copy()

model_df["provider_used"] = model_df["provider_used"].fillna("unknown")

# Missingness indicators used in Week 5
for col in [
    "word_count",
    "char_count",
    "search_volume",
    "competition",
    "cpc",
]:
    model_df[f"has_{col}"] = model_df[col].notna().astype(int)

# Target
y = model_df["ctr_underperforming"].astype(int)

# Groups
groups = model_df["client_id"]

# Same feature set as Week 5
feature_cols_num = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "avg_position",
    "impressions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "days_since_last_update",
    "content_age_days",
    "days_with_impressions",
    "has_word_count",
    "has_char_count",
    "has_search_volume",
    "has_competition",
    "has_cpc",
]

feature_cols_cat = [
    "content_type",
    "main_intent",
    "competition_level",
    "position_tier",
    "provider_used",
    "model_used",
]

X = model_df[feature_cols_num + feature_cols_cat]

# Same preprocessing/model as Week 5
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipeline, feature_cols_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat),
])

def make_model():
    return Pipeline([
        ("prep", preprocess),
        ("lr", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_SEED,
        )),
    ])

def evaluate_predictions(y_true, predictions, probabilities):
    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions),
        "Recall": recall_score(y_true, predictions),
        "F1": f1_score(y_true, predictions),
        "ROC_AUC": roc_auc_score(y_true, probabilities),
    }

# ---------------------------------------------------------
# BEFORE: ordinary row-level stratified cross-validation
# ---------------------------------------------------------

random_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED,
)

random_pred = np.zeros(len(X), dtype=int)
random_proba = np.zeros(len(X))

for train_idx, test_idx in random_cv.split(X, y):

    model = make_model()

    model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    probability = model.predict_proba(
        X.iloc[test_idx]
    )[:, 1]

    random_proba[test_idx] = probability
    random_pred[test_idx] = (
        probability >= 0.5
    ).astype(int)

before = evaluate_predictions(
    y,
    random_pred,
    random_proba
)

# ---------------------------------------------------------
# AFTER: client-grouped cross-validation
# ---------------------------------------------------------

group_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED,
)

group_pred = np.zeros(len(X), dtype=int)
group_proba = np.zeros(len(X))

for train_idx, test_idx in group_cv.split(X, y, groups):

    model = make_model()

    model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    probability = model.predict_proba(
        X.iloc[test_idx]
    )[:, 1]

    group_proba[test_idx] = probability
    group_pred[test_idx] = (
        probability >= 0.5
    ).astype(int)

after = evaluate_predictions(
    y,
    group_pred,
    group_proba
)

# ---------------------------------------------------------
# BEFORE / AFTER TABLE
# ---------------------------------------------------------

comparison = pd.DataFrame([
    {
        "Validation": "Before: random row split",
        **{k: round(v, 3) for k, v in before.items()}
    },
    {
        "Validation": "After: client-grouped split",
        **{k: round(v, 3) for k, v in after.items()}
    }
])

print("Rows:", len(model_df))
print("Clients:", groups.nunique())
print("Positive rate:", round(y.mean(), 3))
print()

comparison

Rows: 28795
Clients: 31
Positive rate: 0.454



,Validation,Accuracy,Precision,Recall,F1,ROC_AUC
0,Before: random row split,0.752,0.736,0.709,0.722,0.834
1,After: client-grouped split,0.731,0.752,0.610,0.674,0.796


### Interpretation

The random row-level split and the client-grouped split were evaluated with
the same model, features, and metrics. The grouped split is the more honest
estimate for this question because pages from the same client can share
patterns. The difference between the two results shows how much the
validation estimate changes when client overlap is removed.

I therefore treat the client-grouped result as the more relevant result for
decision-support. The model's performance should be described as measured on
this dataset and validation design rather than as a guarantee of performance
on future clients.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

The same hunt from Week 3, applied to the final Week-5 feature set. The target
(`ctr_underperforming`) is derived from `ctr` vs `expected_ctr`, so CTR and any
outcome-adjacent columns must not appear as model inputs. Below, every
candidate leakage column is checked against the actual feature set used to
train the model.

The target is constructed from CTR relative to the position-tier median, so
CTR-derived columns are excluded from the model features. The target
construction is also a limitation because the position-tier medians are
calculated from the full dataset rather than learned separately inside each
training fold.

In [5]:
# Leakage audit for the final Week-5 feature set

leakage_candidates = [
    "ctr",
    "clicks",
    "sessions",
    "pageviews",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "ctr_underperforming",
    "expected_ctr",
    "client_id",
    "content_id",
]

feature_set = set(feature_cols_num + feature_cols_cat)

audit_rows = []

for feature in leakage_candidates:

    if feature in feature_set:
        status = "USED"
    else:
        status = "NOT USED"

    if feature in [
        "ctr",
        "clicks",
        "sessions",
        "pageviews",
        "engagement_rate",
        "scroll_rate",
        "ai_traffic_pct",
        "trend_direction",
        "trend_pct",
        "is_declining_label",
        "ctr_underperforming",
        "expected_ctr",
    ]:
        reason = "Outcome/label-derived or closely related to the outcome"
    elif feature == "client_id":
        reason = "Identifier; used for grouping only, not as a feature"
    elif feature == "content_id":
        reason = "Identifier; not used as a model feature"
    else:
        reason = "Not included in final feature set"

    audit_rows.append({
        "Feature": feature,
        "Status": status,
        "Reason": reason
    })

leakage_audit = pd.DataFrame(audit_rows)

leakage_audit

,Feature,Status,Reason
0,ctr,NOT USED,Outcome/label-derived or closely related to th...
1,clicks,NOT USED,Outcome/label-derived or closely related to th...
2,sessions,NOT USED,Outcome/label-derived or closely related to th...
3,pageviews,NOT USED,Outcome/label-derived or closely related to th...
4,engagement_rate,NOT USED,Outcome/label-derived or closely related to th...
5,scroll_rate,NOT USED,Outcome/label-derived or closely related to th...
6,ai_traffic_pct,NOT USED,Outcome/label-derived or closely related to th...
7,trend_direction,NOT USED,Outcome/label-derived or closely related to th...
8,trend_pct,NOT USED,Outcome/label-derived or closely related to th...
9,is_declining_label,NOT USED,Outcome/label-derived or closely related to th...


In [6]:
# Confirm that known leakage candidates are not model features

known_leakage = {
    "ctr",
    "clicks",
    "sessions",
    "pageviews",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "ctr_underperforming",
    "expected_ctr",
}

found_leakage = feature_set.intersection(known_leakage)

print("\nLeakage features found in final feature set:", found_leakage)

if len(found_leakage) == 0:
    print("Result: No known label-derived outcome columns were used as model features.")
else:
    print("WARNING: Review the features listed above.")


Leakage features found in final feature set: set()
Result: No known label-derived outcome columns were used as model features.


In [7]:
# Real error examples from the grouped validation

error_examples = model_df.copy()

error_examples["actual"] = y
error_examples["prediction"] = group_pred
error_examples["probability"] = group_proba

errors = error_examples[
    error_examples["actual"] != error_examples["prediction"]
].copy()

print("\nNumber of grouped-validation errors:", len(errors))

print("\nSample false positives:")
print(
    errors[
        (errors["actual"] == 0) &
        (errors["prediction"] == 1)
    ][
        [
            "avg_position",
            "position_tier",
            "days_with_impressions",
            "content_age_days",
            "actual",
            "prediction",
            "probability"
        ]
    ].head(5)
)

print("\nSample false negatives:")
print(
    errors[
        (errors["actual"] == 1) &
        (errors["prediction"] == 0)
    ][
        [
            "avg_position",
            "position_tier",
            "days_with_impressions",
            "content_age_days",
            "actual",
            "prediction",
            "probability"
        ]
    ].head(5)
)


Number of grouped-validation errors: 7736

Sample false positives:
    avg_position position_tier  days_with_impressions  content_age_days  \
15           7.8        page_1                     17               148   
19           6.9        page_1                     56               187   
23          13.9      striking                     43               502   
27           8.1        page_1                     81               445   
31          42.0      page_3_5                     88               480   

    actual  prediction  probability  
15       0           1     0.925106  
19       0           1     0.643623  
23       0           1     0.577884  
27       0           1     0.609103  
31       0           1     0.519732  

Sample false negatives:
    avg_position position_tier  days_with_impressions  content_age_days  \
5            8.5        page_1                     88               147   
17           7.3        page_1                     88               421   
22 

### Error interpretation

The error examples show that the model does not perfectly separate
underperforming and non-underperforming pages. Some pages with similar
position, visibility, and content characteristics receive different labels.
This suggests that the available features do not capture every factor related
to CTR underperformance.

The model should therefore be used for directional decision-support and
prioritization rather than as an automatic decision rule.

The target is constructed from CTR relative to the position-tier median, so
CTR-derived columns are excluded from the model features. The target
construction is also a limitation because the position-tier medians are
calculated from the full dataset rather than learned separately inside each
training fold.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [8]:
original_claim = (
    "The selected features contain useful information for predicting "
    "CTR underperformance."
)

safer_claim = (
    "The model measured a directional relationship between the selected "
    "features and CTR underperformance in the evaluated dataset. The "
    "results support using the model as decision-support to identify pages "
    "for further review. They do not prove causation or guarantee the same "
    "performance on future or unseen data."
)

print("Original claim:")
print(original_claim)

print("\nSafer claim:")
print(safer_claim)

Original claim:
The selected features contain useful information for predicting CTR underperformance.

Safer claim:
The model measured a directional relationship between the selected features and CTR underperformance in the evaluated dataset. The results support using the model as decision-support to identify pages for further review. They do not prove causation or guarantee the same performance on future or unseen data.


### Claim rewrite

My original Week-5 claim was stronger than the evidence supported. After
reviewing the validation design and leakage risks, I now describe the result
as an observed and measured relationship within the evaluated dataset.

The model is best treated as **directional decision-support**. It can help
identify pages for further review, but it does not prove that the selected
features cause CTR underperformance or guarantee performance on future data.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.